# Math Foundations Part 2 -- Descriptive Stats and Correlation
## Module 2, Class 5

**Objective:** Compute key statistics by hand, then verify with NumPy/Pandas. Understand correlation and what it tells you.

### What you will practice
- Manual mean, variance, standard deviation
- Mean vs median on skewed data
- Correlation matrix and heatmap

---

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("Setup complete.")

## 1. Manual Statistics on a Small Sample

Before using library functions, let's compute everything from scratch on 10 values so you see what the formulas actually do.

In [ ]:
# 10 Sales values (sampled from Superstore)
values = np.array([261.96, 731.94, 14.62, 957.58, 22.37, 
                    48.86, 7.28, 907.15, 18.50, 114.90])

n = len(values)
print(f"Values: {values}")
print(f"n = {n}")

### Mean

$$\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i$$

In [ ]:
# Manual mean
total = np.sum(values)
mean_manual = total / n
print(f"Sum: {total:.2f}")
print(f"Mean (manual): {mean_manual:.4f}")

# Verify
print(f"np.mean():     {np.mean(values):.4f}")
print(f"Match: {np.isclose(mean_manual, np.mean(values))}")

### Variance

$$\sigma^2 = \frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^2$$

Variance measures how spread out the data is from the mean.

In [ ]:
# Manual variance (population variance, ddof=0)
deviations = values - mean_manual
squared_devs = deviations ** 2

print("Value      | Deviation  | Squared")
print("-" * 42)
for v, d, s in zip(values, deviations, squared_devs):
    print(f"{v:10.2f} | {d:10.2f} | {s:12.2f}")

var_manual = np.sum(squared_devs) / n
print(f"\nVariance (manual): {var_manual:.4f}")
print(f"np.var():          {np.var(values):.4f}")
print(f"Match: {np.isclose(var_manual, np.var(values))}")

### Standard Deviation

$$\sigma = \sqrt{\sigma^2}$$

Same unit as the original data, making it easier to interpret than variance.

In [ ]:
# Manual std
std_manual = np.sqrt(var_manual)
print(f"Std (manual): {std_manual:.4f}")
print(f"np.std():     {np.std(values):.4f}")
print(f"Match: {np.isclose(std_manual, np.std(values))}")

print(f"\nInterpretation: On average, values deviate ${std_manual:.0f} from the mean of ${mean_manual:.0f}.")

---
## 2. Mean vs Median on Skewed Data

The mean is sensitive to outliers; the median is not. Let's see this with the full Superstore Sales column.

In [ ]:
# Load Superstore
url = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/superstore.csv"

try:
    df = pd.read_csv(url, encoding='latin-1')
    print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
except Exception as e:
    print(f"URL failed ({e}). Upload your CSV manually.")
    from google.colab import files
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    df = pd.read_csv(filename, encoding='latin-1')
    print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")

# Find sales column
sales_col = [c for c in df.columns if 'sales' in c.lower()][0]
sales = df[sales_col].dropna()

In [ ]:
mean_val = sales.mean()
median_val = sales.median()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(sales, bins=80, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: ${mean_val:,.0f}')
ax.axvline(median_val, color='orange', linestyle='--', linewidth=2, label=f'Median: ${median_val:,.0f}')
ax.set_xlabel('Sales ($)')
ax.set_ylabel('Frequency')
ax.set_title('Sales Distribution: Mean vs Median')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print(f"Mean:   ${mean_val:,.2f}")
print(f"Median: ${median_val:,.2f}")
print(f"Ratio (mean/median): {mean_val/median_val:.2f}x")
print(f"\nThe mean is pulled right by high-value orders. The median is more representative of a 'typical' order.")

**Rule of thumb:**
- Mean > Median --> right-skewed (long tail of high values)
- Mean < Median --> left-skewed (long tail of low values)
- Mean ~= Median --> roughly symmetric

---

## 3. Correlation Matrix

Correlation measures the **linear relationship** between two variables. Range: -1 to +1.

| Value | Meaning |
|-------|---------|
| +1 | Perfect positive linear relationship |
| 0 | No linear relationship |
| -1 | Perfect negative linear relationship |

In [ ]:
# Select numeric columns, drop ID-like columns
num_df = df.select_dtypes(include=[np.number])
id_like = [c for c in num_df.columns if any(kw in c.lower() for kw in ['id', 'postal', 'code', 'zip'])]
num_df = num_df.drop(columns=id_like, errors='ignore')

print(f"Numeric columns: {list(num_df.columns)}")

corr = num_df.corr()
corr.round(3)

In [ ]:
# Heatmap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax,
            vmin=-1, vmax=1)
ax.set_title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Extract strongest correlations (excluding self-correlations)
import itertools

cols = corr.columns.tolist()
pairs_list = []
for i, j in itertools.combinations(range(len(cols)), 2):
    pairs_list.append({
        'Feature 1': cols[i],
        'Feature 2': cols[j],
        'Correlation': corr.iloc[i, j]
    })

pairs_df = pd.DataFrame(pairs_list)
pairs_df['Abs Correlation'] = pairs_df['Correlation'].abs()
pairs_df = pairs_df.sort_values('Abs Correlation', ascending=False)
pairs_df.head(10)

---
## TODO: Interpret the Correlations

Look at the correlation matrix and the ranked pairs above. Answer these questions:

1. Which pair of features has the strongest correlation? Is it positive or negative?
2. Why does that correlation make business sense?
3. If Discount and Profit are negatively correlated, does that mean discounts *cause* lower profits? What's the difference between correlation and causation?

*TODO: Write your answers here.*

1. ...
2. ...
3. ...

---
## Summary

| Concept | Formula | Key point |
|---------|---------|----------|
| Mean | Sum / n | Sensitive to outliers |
| Median | Middle value | Robust to outliers |
| Variance | Avg squared deviation | Measures spread (squared units) |
| Std deviation | Square root of variance | Same units as data |
| Correlation | Pearson coefficient | Linear relationship, not causation |